In [1]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 4.8 MB/s eta 0:00:00


In [2]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [3]:
import os
from pyngrok import ngrok

In [4]:
ngrok.kill()

In [5]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://senior-favorably-fasting.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://senior-favorably-fasting.ngrok-free.dev


True

In [6]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [7]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [8]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Minghsin University of Science and Technology），簡稱明新科大，是一所位於臺灣新竹縣新豐鄉的私立科技大學。學校佔地逾三十公頃，地理位置優越，毗鄰新竹科學園區與新竹工業區，交通便利，擁有豐富的產學合作資源。

該校的創立可追溯至1966年，當時以「明新工業專科學校」之名創校，初期設有機械、土木、工業管理三科。1997年改制為「明新技術學院」並附設專科部。2002年9月，正式升格並更名為「明新科技大學」。

明新科技大學秉持「堅毅、求新、創造」的校訓理念辦學。目前設有半導體學院、工程學院、管理學院、民生學院、人文與設計學院、共同教育學院等六個學院。學校致力於培養跨領域整合、務實創新與全人學習的專業人才，並將自身定位為「一流產業大學」。

明新科大在產業連結方面表現卓越，尤其在半導體領域。學校的半導體學院位處新竹產業核心，並積極與美國、日本、馬來西亞、澳洲及越南等國家的學術單位合作，推動半導體國際人才培育計畫。根據1111人力銀行統計，明新科大畢業生在半導體業界最受歡迎的聘用學校中名列前茅，是唯一入榜的私立科大，與台灣頂尖大學齊名。

此外，明新科技大學也持續創新，整合資源建置「永續智慧商務」教學與實習場域，透過生成式AI與AI專案應用，發展智慧零售、智慧金融、智慧製造與智慧商業等實驗室。學校也積極推動跨領域學習，提供學院間合作的跨領域學程、類產線、2+2N等專班，以提升學生的競爭力。在2022年《遠見》雜誌的「企業最愛公私立技職科大調查」中，明新科大在四大領域中獲得兩項企業最愛，畢業生起薪在私立學校中排名第一，甚至優於部分國立科大.。


In [9]:
result2 = stateful_query("校長是誰？")
print(result2)

明新科技大學現任校長為**呂明峯**教授。他於2025年2月1日正式上任，成為明新科大第11任校長。

呂明峯教授在半導體教育及產業實務方面擁有豐富經驗，並曾任明新科技大學半導體學院院長、工程學院院長、研發長等職務。 他上任後提出了「四大核心模組」作為校務治理策略，致力於將明新科大打造成為新竹地區的人才庫、桃竹苗大矽谷的推動引擎、新南向專班的基地，並活化資源以實現永續校園。


In [ ]:
from flask import Flask, request, abort
import logging
import os
import time
from google.genai import types

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    MessagingApiBlob,
    ReplyMessageRequest,
    TextMessage
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
    FileMessageContent
)

app = Flask(__name__)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
app.logger.setLevel(logging.INFO)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)

# 儲存檔案的目錄
UPLOAD_DIR = "/content/uploaded_files"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# 儲存每個使用者的對話 session 和上傳的檔案
user_sessions = {}  # {user_id: {"chat": chat_object, "uploaded_file": gemini_file}}

def get_user_session(user_id):
    """取得或建立使用者的對話 session"""
    if user_id not in user_sessions:
        # 建立新的對話 session
        new_chat = client.chats.create(
            model="gemini-2.5-flash",
            config=GenerateContentConfig(
                system_instruction="你是一個中文的AI助手，請用繁體中文回答。如果使用者有提供參考文件，請根據文件內容回答問題。",
                tools=[google_search_tool],
                response_modalities=["TEXT"],
            )
        )
        user_sessions[user_id] = {
            "chat": new_chat,
            "uploaded_file": None
        }
    return user_sessions[user_id]

def download_line_file(message_id, file_name):
    """從 LINE 下載使用者上傳的檔案"""
    with ApiClient(configuration) as api_client:
        line_bot_blob_api = MessagingApiBlob(api_client)
        file_content = line_bot_blob_api.get_message_content(message_id)

        file_path = os.path.join(UPLOAD_DIR, file_name)

        with open(file_path, 'wb') as f:
            f.write(file_content)

        return file_path

def upload_file_to_gemini(file_path):
    """上傳檔案到 Gemini Files API"""
    uploaded_file = client.files.upload(
        file=file_path,
        config={'display_name': os.path.basename(file_path)}
    )

    # 等待檔案處理完成
    while uploaded_file.state.name == "PROCESSING":
        print("檔案處理中...")
        time.sleep(1)
        uploaded_file = client.files.get(name=uploaded_file.name)

    if uploaded_file.state.name == "FAILED":
        raise Exception("檔案上傳處理失敗")

    return uploaded_file

def query_with_rag(user_id, question):
    """使用 RAG 模式回答問題"""
    session = get_user_session(user_id)
    uploaded_file = session["uploaded_file"]

    if uploaded_file:
        # 有上傳檔案，使用 RAG 模式
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_uri(
                            file_uri=uploaded_file.uri,
                            mime_type=uploaded_file.mime_type
                        ),
                        types.Part.from_text(text=f"請根據上述提供的檔案內容，用繁體中文回答這個問題：{question}")
                    ]
                )
            ]
        )
        return response.text
    else:
        # 沒有上傳檔案，使用一般多輪對話
        response = session["chat"].send_message(message=question)
        return response.text

@app.route("/", methods=['POST'])
def callback():
    signature = request.headers['X-Line-Signature']
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature.")
        abort(400)

    return 'OK'

@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    """處理文字訊息"""
    text = event.message.text
    user_id = event.source.user_id

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        if text.startswith('AI '):
            prompt = text[3:]
            try:
                # 使用 RAG 或一般對話
                reply_text = query_with_rag(user_id, prompt)

                # 檢查是否有上傳檔案，加上提示
                session = get_user_session(user_id)
                if session["uploaded_file"]:
                    reply_text = f"📄 [RAG 模式]\n\n{reply_text}"

                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=reply_text)]
                    )
                )
            except Exception as e:
                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=f"❌ 發生錯誤：{str(e)}")]
                    )
                )
        elif text == "清除文件":
            # 清除使用者上傳的檔案
            session = get_user_session(user_id)
            session["uploaded_file"] = None
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="✅ 已清除上傳的文件，恢復一般對話模式。")]
                )
            )
        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="請輸入「AI 問題」來開始對話\n或上傳 TXT/PDF 檔案啟用 RAG 模式")]
                )
            )

@handler.add(MessageEvent, message=FileMessageContent)
def handle_file_message(event):
    """處理使用者上傳的檔案"""
    user_id = event.source.user_id
    file_name = event.message.file_name

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        # 檢查檔案類型
        if not (file_name.endswith('.txt') or file_name.endswith('.pdf')):
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="⚠️ 目前只支援 TXT 或 PDF 檔案")]
                )
            )
            return

        try:
            # 下載檔案
            file_path = download_line_file(event.message.id, file_name)
            print(f"檔案已下載：{file_path}")

            # 上傳到 Gemini
            uploaded_file = upload_file_to_gemini(file_path)
            print(f"檔案已上傳到 Gemini：{uploaded_file.uri}")

            # 儲存到使用者 session
            session = get_user_session(user_id)
            session["uploaded_file"] = uploaded_file

            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"✅ 檔案「{file_name}」上傳成功！\n\n現在您可以輸入「AI 問題」來詢問關於這份文件的問題。\n\n輸入「清除文件」可恢復一般對話模式。")]
                )
            )
        except Exception as e:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"❌ 檔案處理失敗：{str(e)}")]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:__main__:Request body: {"destination":"Ub7dc1b532060a415e48eb3b397a1484d","events":[{"type":"message","message":{"type":"text","id":"617040545138606557","quoteToken":"PeoaMobZRmrWi4AOXXIpw1I3dd9cKyYpd1-pQlyi4S3nyKnQlqJ0TEQ4DUIJBGdOWjhwKWzbHbdb2sl1OHHO5FwQn86OLPr98ZbstENjdRB8vJwKp2U8Oz-WKc9olhlN6KizIXkC_nKFDZRRAEwvFQ","markAsReadToken":"v8ca2lwnTY7efw-4QOjDOHXtE7KCqeKWOc8mbUPQVVJELTtCwFE79cFbfdzTgFT6JGwT1CY3RyuO5pPXtdH6Jz4Qt8a70XcpO9EMyaLx2FqhptN65Z6jckm9OpTgSvNCQaEWZewH1tUobVfzHaQHY8EIR8Zm7irWCj7yAHZyxaHOQv4EA3dQc8mw2QUHU7Elh7BtDZn6oNTonXoggytyGA","text":"AI 校長愛吃什麼"},"webhookEventId":"01KTAFVV16S8FEGJ3EWMHPXBM2","deliveryContext":{"isRedelivery":false},"timestamp":1780616129067,"source":{"type":"user","userId":"Ufee3e53621d2838f4ec84bfc9fe80681"},"replyToken":"9fc3fdbef32047c6b1e6

BODY:  {"destination":"Ub7dc1b532060a415e48eb3b397a1484d","events":[{"type":"message","message":{"type":"text","id":"617040545138606557","quoteToken":"PeoaMobZRmrWi4AOXXIpw1I3dd9cKyYpd1-pQlyi4S3nyKnQlqJ0TEQ4DUIJBGdOWjhwKWzbHbdb2sl1OHHO5FwQn86OLPr98ZbstENjdRB8vJwKp2U8Oz-WKc9olhlN6KizIXkC_nKFDZRRAEwvFQ","markAsReadToken":"v8ca2lwnTY7efw-4QOjDOHXtE7KCqeKWOc8mbUPQVVJELTtCwFE79cFbfdzTgFT6JGwT1CY3RyuO5pPXtdH6Jz4Qt8a70XcpO9EMyaLx2FqhptN65Z6jckm9OpTgSvNCQaEWZewH1tUobVfzHaQHY8EIR8Zm7irWCj7yAHZyxaHOQv4EA3dQc8mw2QUHU7Elh7BtDZn6oNTonXoggytyGA","text":"AI 校長愛吃什麼"},"webhookEventId":"01KTAFVV16S8FEGJ3EWMHPXBM2","deliveryContext":{"isRedelivery":false},"timestamp":1780616129067,"source":{"type":"user","userId":"Ufee3e53621d2838f4ec84bfc9fe80681"},"replyToken":"9fc3fdbef32047c6b1e6ce5529878b10","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 23:35:36] "POST / HTTP/1.1" 200 -
INFO:__main__:Request body: {"destination":"Ub7dc1b532060a415e48eb3b397a1484d","events":[{"type":"message","message":{"type":"file","id":"617040679859650561","markAsReadToken":"FnMy1418wg0NAkcYct_xynhZQw51NYDFlXnd1OMNVDm59TgRyVD0g-odwS1_a3bjM7S7yVDQ7_OXc7jJFsLN0ZiZa9gozamfV5s-7GhosJITzF6W-GibCZxmuFK91e0QLRGEer3k-sigBQ7XhSqLYTt66LvjVe564LYN2CrFD39lF7ZxBGFZ06ahAsi5Ti08YJIoGoMfu7seQmxThKXEdA","fileName":"RAG.txt.txt","fileSize":27,"contentProvider":{"type":"line"}},"webhookEventId":"01KTAFY9GZSZ1SH3ZWBNJAVZXM","deliveryContext":{"isRedelivery":false},"timestamp":1780616209445,"source":{"type":"user","userId":"Ufee3e53621d2838f4ec84bfc9fe80681"},"replyToken":"0f5aa3e993b64791ace10aaa1b994d70","mode":"active"}]}


BODY:  {"destination":"Ub7dc1b532060a415e48eb3b397a1484d","events":[{"type":"message","message":{"type":"file","id":"617040679859650561","markAsReadToken":"FnMy1418wg0NAkcYct_xynhZQw51NYDFlXnd1OMNVDm59TgRyVD0g-odwS1_a3bjM7S7yVDQ7_OXc7jJFsLN0ZiZa9gozamfV5s-7GhosJITzF6W-GibCZxmuFK91e0QLRGEer3k-sigBQ7XhSqLYTt66LvjVe564LYN2CrFD39lF7ZxBGFZ06ahAsi5Ti08YJIoGoMfu7seQmxThKXEdA","fileName":"RAG.txt.txt","fileSize":27,"contentProvider":{"type":"line"}},"webhookEventId":"01KTAFY9GZSZ1SH3ZWBNJAVZXM","deliveryContext":{"isRedelivery":false},"timestamp":1780616209445,"source":{"type":"user","userId":"Ufee3e53621d2838f4ec84bfc9fe80681"},"replyToken":"0f5aa3e993b64791ace10aaa1b994d70","mode":"active"}]}
檔案已下載：/content/uploaded_files/RAG.txt.txt
檔案已上傳到 Gemini：https://generativelanguage.googleapis.com/v1beta/files/qtz6qo4u99ur


INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 23:36:52] "POST / HTTP/1.1" 200 -
INFO:__main__:Request body: {"destination":"Ub7dc1b532060a415e48eb3b397a1484d","events":[{"type":"message","message":{"type":"text","id":"617040692945879602","quoteToken":"xFR2sRDUb7kHUZ3ZB4Ls9ePBMrlUnNnwlzDud1uVKwQ_tAoVfjcsfs6rpX4-TzpTRuub0wcd_bF_VmwfA8nqi_nJEu4bXj_EoLcpM2ODyMx6pzaqB_U3oCpNBhP9Zz813VM7ENtoM30kwc6pGFigjA","markAsReadToken":"I-o9gp3hpCajUwZL6m2DQ6cXwZAew4XhA493Ivw9gf83cbPXpJjM8QwajnDraqAEvg08g6v_JNRdRSXUbhSbm0fvrWZJRgVmp3HPdxCppdGZtCUIZW-FbLCf9WNZZUZ9c4cDYoZmP6jGb4sfaIrVO63FnMhkgBI2iJft6U8hgCwvc6fQWrhVB6b7wCPR8i5yPRGtQWolDytbksrzCXhDJw","text":"AI 校長愛吃什麼"},"webhookEventId":"01KTAFYH2CKA7W6RJSTP8AMHTB","deliveryContext":{"isRedelivery":false},"timestamp":1780616217168,"source":{"type":"user","userId":"Ufee3e53621d2838f4ec84bfc9fe80681"},"replyToken":"bcc700412f2f47a4a0ee964783bb6955","mode":"active"}]}


BODY:  {"destination":"Ub7dc1b532060a415e48eb3b397a1484d","events":[{"type":"message","message":{"type":"text","id":"617040692945879602","quoteToken":"xFR2sRDUb7kHUZ3ZB4Ls9ePBMrlUnNnwlzDud1uVKwQ_tAoVfjcsfs6rpX4-TzpTRuub0wcd_bF_VmwfA8nqi_nJEu4bXj_EoLcpM2ODyMx6pzaqB_U3oCpNBhP9Zz813VM7ENtoM30kwc6pGFigjA","markAsReadToken":"I-o9gp3hpCajUwZL6m2DQ6cXwZAew4XhA493Ivw9gf83cbPXpJjM8QwajnDraqAEvg08g6v_JNRdRSXUbhSbm0fvrWZJRgVmp3HPdxCppdGZtCUIZW-FbLCf9WNZZUZ9c4cDYoZmP6jGb4sfaIrVO63FnMhkgBI2iJft6U8hgCwvc6fQWrhVB6b7wCPR8i5yPRGtQWolDytbksrzCXhDJw","text":"AI 校長愛吃什麼"},"webhookEventId":"01KTAFYH2CKA7W6RJSTP8AMHTB","deliveryContext":{"isRedelivery":false},"timestamp":1780616217168,"source":{"type":"user","userId":"Ufee3e53621d2838f4ec84bfc9fe80681"},"replyToken":"bcc700412f2f47a4a0ee964783bb6955","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 23:36:59] "POST / HTTP/1.1" 200 -
